<a href="https://colab.research.google.com/github/GunaPalanivel/Praxis/blob/main/praxis_grpo_colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Praxis GRPO Training — Judge Evidence Notebook

**What this runs:** A GRPO-style policy gradient loop over the Praxis SRE incident environment.
A softmax policy learns to select the correct investigation/remediation sequence by maximising
group-relative advantages from live environment rewards.

**Runtime:** ~30–60 min on Colab CPU/T4 (no GPU required).

**Outputs saved for judges:**
- `praxis_training_log.jsonl` — per-episode reward, loss, actions, task
- `praxis_training_summary.csv` — tabular summary by episode
- `reward_curve.png` — reward + eval checkpoint comparison
- `loss_curve.png` — GRPO loss curve
- `rollout_trace.txt` — structured [START]/[STEP]/[END] rollout log
- `eval_checkpoints.json` — baseline vs trained scores per task

**Environment:** `https://gp5901-praxis.hf.space` (live HF Space)


In [ ]:
# Cell 1 — Install deps (trl, matplotlib, numpy, requests)
!pip install -q trl matplotlib numpy requests

In [ ]:
# Cell 2 — Config
import math, os, random, time, json, csv
from dataclasses import dataclass, asdict
from datetime import datetime
import numpy as np

PRAXIS_BASE_URL = "https://gp5901-praxis.hf.space"

TASKS = [
    "single-service-alert",
    "ambiguous-incident",
    "cascading-failure",
    "memory-leak",
]

SEED = 2026
random.seed(SEED)
np.random.seed(SEED)

@dataclass
class TrainConfig:
    episodes: int = 80
    group_size: int = 4
    max_steps: int = 10
    lr: float = 0.06
    step_delay: float = 0.4   # seconds between API calls — avoids 429 rate limit
    eval_every: int = 10
    eval_episodes: int = 5

cfg = TrainConfig()
est_min = cfg.episodes * cfg.group_size * cfg.max_steps * cfg.step_delay / 60
print(f"Config: {cfg}")
print(f"Est. API calls: {cfg.episodes * cfg.group_size * cfg.max_steps}")
print(f"Est. runtime: ~{est_min:.0f} min")


In [ ]:
# Cell 3 — API helpers with retry + exponential backoff on 429
import requests

def api_get(path: str, retries: int = 3) -> dict:
    for attempt in range(retries):
        try:
            r = requests.get(f"{PRAXIS_BASE_URL}{path}", timeout=30)
            r.raise_for_status()
            return r.json()
        except requests.HTTPError as e:
            if r.status_code == 429:
                wait = 2 ** attempt * 5
                print(f"  [429] waiting {wait}s...")
                time.sleep(wait)
            else:
                raise
    raise RuntimeError(f"GET {path} failed after {retries} retries")

def api_post(path: str, payload: dict, headers: dict = None, retries: int = 3) -> dict:
    for attempt in range(retries):
        try:
            r = requests.post(
                f"{PRAXIS_BASE_URL}{path}",
                json=payload,
                headers=headers or {},
                timeout=30,
            )
            r.raise_for_status()
            return r.json()
        except requests.HTTPError as e:
            if r.status_code == 429:
                wait = 2 ** attempt * 5
                print(f"  [429] waiting {wait}s...")
                time.sleep(wait)
            else:
                raise
    raise RuntimeError(f"POST {path} failed after {retries} retries")

health = api_get("/health")
print("health:", health)
print("Space is live ✓")


In [ ]:
# Cell 4 — Per-task action pools
ACTION_POOLS = {
    "single-service-alert": [
        "query_logs service=auth timerange=5m",
        "check_metrics service=auth metric=error_rate",
        "check_config service=auth",
        "check_runbook service=auth",
        "diagnose root_cause=bad_config",
        "diagnose root_cause=db_connection_failure",
        "rollback_deploy service=auth",
        "restart_service service=auth",
        "escalate reason=need_senior_support",
    ],
    "ambiguous-incident": [
        "query_logs service=app timerange=10m",
        "query_logs service=dns-resolver timerange=10m",
        "check_metrics service=dns-resolver metric=resolution_failures",
        "check_metrics service=app metric=error_rate",
        "check_config service=dns-resolver",
        "check_deps service=app",
        "check_runbook service=dns-resolver",
        "diagnose root_cause=dns_misconfiguration",
        "restart_service service=dns-resolver",
        "escalate reason=cross_service_incident",
    ],
    "cascading-failure": [
        "query_logs service=api timerange=10m",
        "query_logs service=database timerange=15m",
        "query_logs service=analytics timerange=10m",
        "check_metrics service=database metric=connections",
        "check_deps service=api",
        "check_config service=database",
        "check_runbook service=database",
        "diagnose root_cause=db_connection_pool_exhausted",
        "kill_query service=database query_id=runaway_analytics",
        "scale_resource service=database resource=connection_pool",
        "escalate reason=db_overload",
    ],
    "memory-leak": [
        "query_logs service=worker timerange=10m",
        "check_metrics service=worker metric=memory",
        "check_config service=worker",
        "check_runbook service=worker",
        "diagnose root_cause=large_batch_size_oom",
        "diagnose root_cause=memory_leak",
        "rollback_deploy service=worker",
        "scale_resource service=worker resource=memory",
        "escalate reason=oom_crash",
    ],
}

POLICY_LOGITS = {
    task: np.zeros(len(actions), dtype=np.float64)
    for task, actions in ACTION_POOLS.items()
}
print("Action pool sizes:", {t: len(a) for t, a in ACTION_POOLS.items()})


In [ ]:
# Cell 5 — Policy and episode runner
def softmax(logits: np.ndarray) -> np.ndarray:
    x = logits - np.max(logits)
    e = np.exp(x)
    return e / np.sum(e)

def run_episode(task: str, logits: np.ndarray, max_steps: int = 10, delay: float = 0.4):
    actions = ACTION_POOLS[task]
    reset_resp = api_post("/reset", {"task_name": task})
    sid = reset_resp.get("session_id", "")
    headers = {"x-session-id": sid} if sid else {}

    rewards, chosen_idxs, trace_lines = [], [], []
    trace_lines.append(f"[START] task={task} env=praxis model=softmax-grpo")

    for step in range(1, max_steps + 1):
        probs = softmax(logits)
        idx = int(np.random.choice(len(actions), p=probs))
        command = actions[idx]

        time.sleep(delay)  # rate limit guard
        resp = api_post("/step", {"command": command}, headers=headers)
        reward = float(resp["reward"])
        done = bool(resp["done"])

        rewards.append(reward)
        chosen_idxs.append(idx)
        trace_lines.append(
            f"[STEP] step={step} action={command} "
            f"reward={reward:.2f} done={str(done).lower()} error=null"
        )
        if done:
            break

    score = float(np.mean(rewards)) if rewards else 0.01
    success = score >= 0.15
    trace_lines.append(
        f"[END] success={str(success).lower()} steps={len(rewards)} "
        f"score={score:.3f} rewards={','.join(f'{r:.2f}' for r in rewards)}"
    )
    return rewards, chosen_idxs, trace_lines

print("Episode runner ready.")


In [ ]:
# Cell 6 — Baseline eval (untrained policy)
print("Running baseline evaluation (untrained policy)...")
baseline_scores = {}

for task in TASKS:
    task_rewards = []
    for ep in range(cfg.eval_episodes):
        rewards, _, _ = run_episode(task, POLICY_LOGITS[task], max_steps=cfg.max_steps, delay=cfg.step_delay)
        if rewards:
            task_rewards.append(float(np.mean(rewards)))
    baseline_scores[task] = float(np.mean(task_rewards)) if task_rewards else 0.0
    print(f"  baseline [{task}]: {baseline_scores[task]:.4f}")

print(f"\nBaseline mean: {np.mean(list(baseline_scores.values())):.4f}")


In [ ]:
# Cell 7 — GRPO training loop (~30-60 min)
print(f"Starting GRPO training: {cfg.episodes} episodes, group_size={cfg.group_size}")
print(f"Tasks: {TASKS}")
print("=" * 60)

training_log = []
reward_curve = []
loss_curve = []
eval_checkpoints = []
all_traces = []

start_time = time.time()

for ep_idx in range(cfg.episodes):
    task = TASKS[ep_idx % len(TASKS)]
    logits = POLICY_LOGITS[task]
    actions = ACTION_POOLS[task]

    group_returns, group_action_seqs = [], []
    for g in range(cfg.group_size):
        rewards, chosen_idxs, trace = run_episode(
            task, logits, max_steps=cfg.max_steps, delay=cfg.step_delay
        )
        ep_return = float(np.mean(rewards)) if rewards else 0.01
        group_returns.append(ep_return)
        group_action_seqs.append(chosen_idxs)
        if g == 0:
            all_traces.extend(trace)
            all_traces.append("")

    returns_arr = np.array(group_returns, dtype=np.float64)
    baseline_val = float(np.mean(returns_arr))
    std = float(np.std(returns_arr) + 1e-8)
    advantages = (returns_arr - baseline_val) / std

    probs = softmax(logits)
    grad = np.zeros_like(logits)
    loss = 0.0
    total_actions = max(1, sum(len(a) for a in group_action_seqs))

    for adv, act_seq in zip(advantages, group_action_seqs):
        for act in act_seq:
            one_hot = np.zeros_like(logits)
            one_hot[act] = 1.0
            grad += adv * (one_hot - probs)
            loss += -adv * math.log(max(probs[act], 1e-8))

    grad /= total_actions
    loss /= total_actions
    POLICY_LOGITS[task] += cfg.lr * grad

    mean_ep_reward = float(np.mean(group_returns))
    reward_curve.append(mean_ep_reward)
    loss_curve.append(float(loss))

    record = {
        "episode": ep_idx + 1,
        "task": task,
        "mean_reward": round(mean_ep_reward, 4),
        "grpo_loss": round(float(loss), 6),
        "group_returns": [round(r, 4) for r in group_returns],
        "baseline_val": round(baseline_val, 4),
        "timestamp": datetime.utcnow().isoformat(),
        "elapsed_sec": round(time.time() - start_time, 1),
    }
    training_log.append(record)

    if (ep_idx + 1) % cfg.eval_every == 0:
        eval_r = []
        for _ in range(cfg.eval_episodes):
            r, _, _ = run_episode(task, POLICY_LOGITS[task], max_steps=cfg.max_steps, delay=cfg.step_delay)
            if r:
                eval_r.append(float(np.mean(r)))
        eval_score = float(np.mean(eval_r)) if eval_r else 0.0
        eval_checkpoints.append({"episode": ep_idx + 1, "task": task, "eval_score": round(eval_score, 4)})
        elapsed = (time.time() - start_time) / 60
        print(
            f"  ep {ep_idx+1:3d}/{cfg.episodes} | task={task:22s} | "
            f"train_r={mean_ep_reward:.4f} | eval_r={eval_score:.4f} | "
            f"loss={loss:.4f} | {elapsed:.1f}min"
        )

total_time = (time.time() - start_time) / 60
print(f"\nTraining complete in {total_time:.1f} min.")
print(f"Final reward (last 10 ep): {np.mean(reward_curve[-10:]):.4f}")


In [ ]:
# Cell 8 — Post-training eval
print("Post-training evaluation...")
trained_scores = {}

for task in TASKS:
    task_rewards = []
    for ep in range(cfg.eval_episodes):
        rewards, _, _ = run_episode(task, POLICY_LOGITS[task], max_steps=cfg.max_steps, delay=cfg.step_delay)
        if rewards:
            task_rewards.append(float(np.mean(rewards)))
    trained_scores[task] = float(np.mean(task_rewards)) if task_rewards else 0.0
    delta = trained_scores[task] - baseline_scores[task]
    print(f"  trained [{task}]: {trained_scores[task]:.4f}  (Δ {delta:+.4f})")

bm = np.mean(list(baseline_scores.values()))
tm = np.mean(list(trained_scores.values()))
print(f"\nBaseline mean: {bm:.4f}")
print(f"Trained  mean: {tm:.4f}")
print(f"Improvement  : {tm-bm:+.4f} ({(tm/max(bm,1e-8)-1)*100:+.1f}%)")


In [ ]:
# Cell 9 — Save all judge-ready artifacts
import csv as csv_mod

with open("praxis_training_log.jsonl", "w") as f:
    for rec in training_log:
        f.write(json.dumps(rec) + "\n")
print(f"Saved praxis_training_log.jsonl ({len(training_log)} records)")

with open("praxis_training_summary.csv", "w", newline="") as f:
    fieldnames = ["episode", "task", "mean_reward", "grpo_loss", "baseline_val", "elapsed_sec", "timestamp"]
    writer = csv_mod.DictWriter(f, fieldnames=fieldnames)
    writer.writeheader()
    for rec in training_log:
        writer.writerow({k: rec[k] for k in fieldnames})
print("Saved praxis_training_summary.csv")

with open("rollout_trace.txt", "w") as f:
    f.write("\n".join(all_traces))
print(f"Saved rollout_trace.txt ({len(all_traces)} lines)")

with open("eval_checkpoints.json", "w") as f:
    json.dump({
        "baseline_scores": baseline_scores,
        "trained_scores": trained_scores,
        "checkpoints": eval_checkpoints,
        "config": asdict(cfg),
        "total_episodes": len(training_log),
    }, f, indent=2)
print("Saved eval_checkpoints.json")


In [ ]:
# Cell 10 — Plot reward + loss curves
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec

def moving_avg(data, window=5):
    return np.convolve(data, np.ones(window)/window, mode='valid')

fig = plt.figure(figsize=(14, 10))
gs = gridspec.GridSpec(2, 2, figure=fig)
episodes_x = np.arange(1, len(reward_curve) + 1)

ax1 = fig.add_subplot(gs[0, :])
ax1.plot(episodes_x, reward_curve, color='#aac4e6', alpha=0.5, linewidth=1, label='Per-episode reward')
if len(reward_curve) >= 5:
    ma = moving_avg(reward_curve, window=5)
    ax1.plot(np.arange(5, len(reward_curve)+1), ma, color='#2563eb', linewidth=2.5, label='Moving avg (w=5)')
if eval_checkpoints:
    ckpt_ep = [c['episode'] for c in eval_checkpoints]
    ckpt_sc = [c['eval_score'] for c in eval_checkpoints]
    ax1.scatter(ckpt_ep, ckpt_sc, color='#16a34a', s=60, zorder=5, label='Eval checkpoint')
bm = np.mean(list(baseline_scores.values()))
ax1.axhline(y=bm, color='#dc2626', linestyle='--', linewidth=1.5, label=f'Baseline mean ({bm:.3f})')
ax1.set_xlabel('Episode', fontsize=12)
ax1.set_ylabel('Mean Episode Reward', fontsize=12)
ax1.set_title('Praxis GRPO Training — Reward Curve', fontsize=14, fontweight='bold')
ax1.legend(fontsize=10)
ax1.grid(True, alpha=0.3)

ax2 = fig.add_subplot(gs[1, 0])
ax2.plot(episodes_x, loss_curve, color='#7c3aed', linewidth=1.5)
ax2.set_xlabel('Episode'); ax2.set_ylabel('GRPO Loss')
ax2.set_title('GRPO Loss Curve', fontsize=12, fontweight='bold')
ax2.grid(True, alpha=0.3)

ax3 = fig.add_subplot(gs[1, 1])
x = np.arange(len(TASKS)); w = 0.35
b_vals = [baseline_scores[t] for t in TASKS]
t_vals = [trained_scores[t] for t in TASKS]
ax3.bar(x - w/2, b_vals, w, label='Baseline', color='#ef4444', alpha=0.8)
ax3.bar(x + w/2, t_vals, w, label='Trained', color='#22c55e', alpha=0.8)
ax3.set_xticks(x)
ax3.set_xticklabels([t[:18] for t in TASKS], rotation=15, ha='right', fontsize=9)
ax3.set_ylabel('Mean Reward')
ax3.set_title('Baseline vs Trained (per task)', fontsize=12, fontweight='bold')
ax3.legend(fontsize=9); ax3.grid(True, alpha=0.3, axis='y')

plt.suptitle(f'Praxis GRPO — {len(training_log)} episodes | G={cfg.group_size} | LR={cfg.lr}', fontsize=13)
plt.tight_layout()
plt.savefig('reward_curve.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved reward_curve.png")

fig2, ax = plt.subplots(figsize=(10, 4))
ax.plot(episodes_x, loss_curve, color='#7c3aed', linewidth=1.5)
ax.set_xlabel('Episode'); ax.set_ylabel('GRPO Loss')
ax.set_title('Praxis GRPO Training Loss'); ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('loss_curve.png', dpi=150)
plt.show()
print("Saved loss_curve.png")


In [ ]:
# Cell 11 — Judge summary table
print("=" * 65)
print("JUDGE SUMMARY — Praxis GRPO Training Run")
print("=" * 65)
print(f"{'Task':<28} {'Baseline':>10} {'Trained':>10} {'Delta':>10}")
print("-" * 65)
for task in TASKS:
    b = baseline_scores[task]; t = trained_scores[task]; d = t - b
    print(f"{task:<28} {b:>10.4f} {t:>10.4f} {d:>+10.4f}")
print("-" * 65)
bm = np.mean(list(baseline_scores.values()))
tm = np.mean(list(trained_scores.values()))
print(f"{'MEAN':<28} {bm:>10.4f} {tm:>10.4f} {tm-bm:>+10.4f}")
print("=" * 65)
print(f"\nTotal episodes trained : {len(training_log)}")
print(f"Group size (G)         : {cfg.group_size}")
print(f"Learning rate          : {cfg.lr}")
print(f"Max steps/episode      : {cfg.max_steps}")
print(f"\nArtifacts for judges:")
print("  praxis_training_log.jsonl     — full per-episode log")
print("  praxis_training_summary.csv   — tabular reward/loss")
print("  rollout_trace.txt             — [START]/[STEP]/[END] traces")
print("  eval_checkpoints.json         — baseline vs trained scores")
print("  reward_curve.png              — reward + eval plot")
print("  loss_curve.png                — GRPO loss curve")


In [ ]:
# Cell 12 — Download all artifacts to local machine
try:
    from google.colab import files
    for fname in [
        "praxis_training_log.jsonl",
        "praxis_training_summary.csv",
        "eval_checkpoints.json",
        "rollout_trace.txt",
        "reward_curve.png",
        "loss_curve.png",
    ]:
        files.download(fname)
    print("All artifacts downloaded.")
except ImportError:
    print("Not in Colab — files saved to working directory.")
